# 16. 과거 시기 데이터 — 2022–23년 위성 사진을 v1과 똑같이 만들기 (실험 7 준비)

**한 줄 요약:** v1(2024–25)과 **기간만 2년 앞당긴** 2022–23년 여름·낙엽기 중앙값 20밴드를 GEE로 내보내고, **v1 캐시와 같은 픽셀**에서 38개 피처를 05와 똑같은 방법으로 계산해 `{지역}_old.npz`로 저장한다. 그 사이 숲이 베어진 픽셀을 거르려고 **Hansen 산림손실 연도** 층도 같이 받는다.

**쉬운 설명**
- 실험 7 질문: 같은 숲을 2년 간격으로 두 번 찍어 같은 모델로 탄소를 계산하면 얼마나 다르게 나오나 → 숲이 그대로인데 숫자가 흔들리면, 그 흔들림이 "가짜 변화량"
- 그래서 사진만 바꾸고 나머지(구름 처리, 밴드, 월, 격자, 피처 계산)는 v1과 한 글자도 다르지 않게 만든다
- 피처 계산 코드가 05와 같은지 **v1 영상으로 먼저 검산**한다 (v1 캐시 값과 똑같이 나와야 통과)

**12번과 같은 방식:** 2곳씩 ① 제출 → ② 상태 확인 → ③ 이동·캐시·원본 삭제 → ④ Drive 휴지통 비우기. **"모두 실행" 하지 말 것** (8곳이 한꺼번에 제출됨)

**v1 영상(`{지역}_s2_20band.tif`)은 지우지 않는다.** 검산에 필요하다.

**순서:** [1] 설정 → [2] 함수 → [3] 용량 확인 → 묶음 A~D → [7] 진행표

In [ ]:
# [1] 설정 — 02·12와 같은 GEE 프로젝트·경로
!pip install -q earthengine-api rasterio

import ee, os, glob, json, time, gc, shutil
import numpy as np
import rasterio
from rasterio.windows import Window
from scipy.ndimage import uniform_filter, distance_transform_edt
from google.colab import drive

PROJECT_ID = 'driven-era-467023-m3'
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT_ID)

drive.mount('/content/drive')
D       = '/content/drive/MyDrive/forest/data'
OUT     = '/content/drive/MyDrive/forest/outputs'
SRC_DIR = '/content/drive/MyDrive/forest_s2'
LOG     = f'{OUT}/old_export_log.json'

TRAIN_REGIONS = ['inje', 'hongcheon', 'gangneung', 'pyeongchang',
                 'bonghwa', 'geumsan', 'geochang', 'suncheon']

START_OLD, END_OLD = '2022-01-01', '2023-12-31'      # v1은 2024-01-01 ~ 2025-12-31 (2년 앞당김, 나머지 같음)
BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']   # v1과 같음
SUMMER, LEAFOFF = [6, 7, 8, 9], [11, 12, 1, 2]                            # v1과 같음
HANSEN_IDS = ['UMD/hansen/global_forest_change_2024_v1_12', 'UMD/hansen/global_forest_change_2023_v1_11']

BLOCK, PAD, EDGE_PX = 1024, 16, 7      # 05 extract_samples와 같음
VERIFY_BLOCKS = 6                      # v1 영상으로 피처 코드 검산할 블록 수
DELETE_RAW = True

for r in TRAIN_REGIONS:
    assert os.path.exists(f'{D}/s2/{r}_s2_20band.tif'), f'v1 영상 없음: {r} (지우면 안 됨)'
    assert os.path.exists(f'{OUT}/{r}_s05.npz') and os.path.exists(f'{OUT}/{r}_v2.npz'), f'캐시 없음: {r}'
print('v1 영상·캐시 8곳 확인')

Mounted at /content/drive
v1 영상·캐시 8곳 확인


In [ ]:
# [2] 함수
def cloud_mask(img):
    # 02와 같음: SCL 3(그림자) 8·9(구름) 10(권운) 제외
    scl = img.select('SCL')
    good = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(good).divide(10000)

def composite(aoi, months, cloud_pct=50):
    # 02 composite_aoi와 같음 (기간만 START_OLD~END_OLD)
    mf = ee.Filter.Or([ee.Filter.calendarRange(m, m, 'month') for m in months])
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(aoi).filterDate(START_OLD, END_OLD).filter(mf)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_pct)))
    return col.map(cloud_mask).median(), col.size()

def hansen_lossyear():
    for hid in HANSEN_IDS:
        try:
            img = ee.Image(hid); img.bandNames().getInfo()
            return img.select('lossyear'), hid
        except Exception:
            continue
    raise RuntimeError('Hansen 산림손실 자료를 찾지 못함 — 멈추고 보고')

def v1_grid(name):
    with rasterio.open(f'{D}/s2/{name}_s2_20band.tif') as s:
        epsg = s.crs.to_epsg()
        assert epsg is not None, f'{name}: v1 좌표계 EPSG 없음'
        return dict(crs=f'EPSG:{epsg}', tr=s.transform, b=s.bounds, H=s.height, W=s.width)

def submit_old(name):
    g = v1_grid(name)
    aoi = ee.Geometry.Rectangle([g['b'].left, g['b'].bottom, g['b'].right, g['b'].top], proj=g['crs'], geodesic=False)
    s_img, n_s = composite(aoi, SUMMER)
    w_img, n_w = composite(aoi, LEAFOFF)
    stack = (s_img.select(BANDS).rename([f's_{b}' for b in BANDS])
             .addBands(w_img.select(BANDS).rename([f'w_{b}' for b in BANDS])))
    stack = stack.multiply(10000).toInt16().unmask(0)          # v1과 같은 변환 (값 없음 = 0)
    loss, hid = hansen_lossyear()
    stack = stack.addBands(loss.unmask(0).toInt16().rename('lossyear'))
    tr = g['tr']
    task = ee.batch.Export.image.toDrive(
        image=stack, description=f'{name}_s2old', folder='forest_s2',
        fileNamePrefix=f'{name}_s2old_21band', region=aoi, crs=g['crs'],
        crsTransform=[tr.a, tr.b, tr.c, tr.d, tr.e, tr.f], maxPixels=1e10)
    task.start()
    counts = dict(summer=int(n_s.getInfo()), leafoff=int(n_w.getInfo()))
    log = json.load(open(LOG)) if os.path.exists(LOG) else {}
    log[name] = dict(submitted=time.strftime('%Y-%m-%d %H:%M'), scenes=counts, hansen=hid, crs=g['crs'], H=g['H'], W=g['W'])
    json.dump(log, open(LOG, 'w'), ensure_ascii=False, indent=2)
    print(f'{name:12s} 제출  장면 수 {counts}  Hansen {hid.split("/")[-1]}  격자 {g["H"]}x{g["W"]}')

def make_feats(X):
    # 05 make_feats와 한 글자도 같게 — (20,h,w) 반사율 → (38,h,w)
    nd = lambda a, b: (a - b) / (a + b + 1e-6)
    s_ndvi, w_ndvi = nd(X[6], X[2]),  nd(X[16], X[12])
    s_ndmi, w_ndmi = nd(X[6], X[8]),  nd(X[16], X[18])
    d_ndvi, d_ndmi = s_ndvi - w_ndvi, s_ndmi - w_ndmi
    f = np.concatenate([X, np.stack([s_ndvi, w_ndvi, s_ndmi, w_ndmi, d_ndvi, d_ndmi])])
    extra = []
    for base in (s_ndvi, w_ndvi, d_ndvi):
        for w in (5, 15):
            a  = np.nan_to_num(base)
            mu = uniform_filter(a, size=w)
            sd = np.sqrt(np.maximum(uniform_filter(a**2, size=w) - mu**2, 0))
            extra += [mu, sd]
    return np.concatenate([f, np.stack(extra)]).astype('float32')

def build_old_cache(name, old_path):
    t0 = time.time()
    z2 = np.load(f'{OUT}/{name}_v2.npz')                 # 픽셀 위치 (v1 캐시와 같은 순서, 12에서 검산됨)
    rows, cols, y, tile = z2['rows'], z2['cols'], z2['y'], z2['tile']
    z1 = np.load(f'{OUT}/{name}_s05.npz')
    X1 = z1['X']
    assert len(y) == len(z1['y']) and (y == z1['y']).all(), f'{name}: v1 캐시와 픽셀 다름'
    n = len(y)

    with rasterio.open(f'{D}/s2/{name}_s2_20band.tif') as s1, rasterio.open(old_path) as s2:
        H, W = s1.height, s1.width
        assert s2.crs == s1.crs and s2.count == 21, f'{name}: 과거 영상 좌표계/밴드 수 이상 ({s2.count})'
        t1, t2 = s1.transform, s2.transform
        assert abs(t2.a - t1.a) < 1e-6 and abs(t2.e - t1.e) < 1e-6, f'{name}: 픽셀 크기 다름'
        dc, dr = (t1.c - t2.c) / t1.a, (t1.f - t2.f) / t1.e
        assert abs(dc - round(dc)) < 1e-3 and abs(dr - round(dr)) < 1e-3, f'{name}: 격자 어긋남 {dc:.3f},{dr:.3f}'
        co, ro = int(round(dc)), int(round(dr))
        rd2 = lambda c0, r0, w, h, bands: s2.read(bands, window=Window(c0 + co, r0 + ro, w, h), boundless=True, fill_value=0)

        # 과거 영상의 결측 경계 (05 image_ok_mask와 같은 규칙, v1 격자 기준)
        valid = np.zeros((H, W), bool)
        for r0 in range(0, H, 2048):
            for c0 in range(0, W, 2048):
                h, w = min(2048, H - r0), min(2048, W - c0)
                valid[r0:r0 + h, c0:c0 + w] = rd2(c0, r0, w, h, list(range(1, 21))).sum(0, dtype='float32') > 0
        edge_ok = valid if valid.all() else distance_transform_edt(valid) >= EDGE_PX
        edge_ok = edge_ok[rows, cols]; del valid; gc.collect()

        # 05와 같은 블록·패딩으로 피처 계산 → 표본 픽셀에서 뽑기
        nbc = (W + BLOCK - 1) // BLOCK
        bidx = (rows // BLOCK) * nbc + cols // BLOCK
        assert (np.diff(bidx) >= 0).all(), f'{name}: 표본 순서가 05 블록 순서와 다름'
        Xo = np.full((n, 38), np.nan, 'float32'); raw_ok = np.zeros(n, bool); loss = np.zeros(n, 'int16')
        vdiff, nver = 0.0, 0
        for b in np.unique(bidx):
            i0, i1 = np.searchsorted(bidx, b), np.searchsorted(bidx, b, 'right')
            idx = np.arange(i0, i1)
            r0, c0 = (b // nbc) * BLOCK, (b % nbc) * BLOCK
            r1, c1 = min(H, r0 + BLOCK), min(W, c0 + BLOCK)
            pr0, pc0, pr1, pc1 = max(0, r0 - PAD), max(0, c0 - PAD), min(H, r1 + PAD), min(W, c1 + PAD)
            lr, lc = rows[idx] - pr0, cols[idx] - pc0
            R = rd2(pc0, pr0, pc1 - pc0, pr1 - pr0, list(range(1, 22)))
            raw_ok[idx] = (R[:20, lr, lc] > 0).all(0)
            loss[idx] = R[20, lr, lc]
            Xo[idx] = make_feats(R[:20].astype('float32') / 10000.0)[:, lr, lc].T
            del R
            if nver < VERIFY_BLOCKS:      # 같은 코드로 v1 영상 → v1 캐시 값과 같아야 함
                R1 = s1.read(window=Window(pc0, pr0, pc1 - pc0, pr1 - pr0)).astype('float32') / 10000.0
                vdiff = max(vdiff, float(np.abs(make_feats(R1)[:, lr, lc].T - X1[idx]).max()))
                nver += 1; del R1
    assert vdiff < 1e-4, f'{name}: 피처 계산이 05와 다름 (최대 차이 {vdiff:.2e}) — 멈추고 보고'

    ok = raw_ok & edge_ok
    Xo[~ok] = np.nan
    np.savez_compressed(f'{OUT}/{name}_old.npz', X=Xo, ok=ok, loss=loss, y=y, tile=tile,
                        period=np.array(f'{START_OLD}~{END_OLD}'), verify_maxdiff=np.float32(vdiff))
    zz = np.load(f'{OUT}/{name}_old.npz'); assert zz['X'].shape == (n, 38)
    lm = (loss >= 22)
    print(f'{name:12s} 캐시 저장 {n:,}픽셀, 값 있음 {ok.mean()*100:.1f}%, 2022년 이후 산림손실 {lm.mean()*100:.2f}%, '
          f'피처 검산 차이 {vdiff:.1e} ({time.time()-t0:.0f}초)')

# ---- 묶음 실행용 (12와 같은 모양) ----
def submit_pair(now):
    for r in now:
        if os.path.exists(f'{OUT}/{r}_old.npz'):
            print(f'{r:12s} 캐시 있음 — 건너뜀'); continue
        if glob.glob(f'{SRC_DIR}/{r}_s2old_21band*.tif') or glob.glob(f'{D}/s2/{r}_s2old_21band*.tif'):
            print(f'{r:12s} 영상이 이미 Drive에 있음 — ③으로'); continue
        submit_old(r)
    print('\n→ ② 상태 확인 (지역당 수십 분 [추정])')

def status_pair(now):
    want, seen = {f'{r}_s2old' for r in now}, set()
    for t in ee.batch.Task.list()[:30]:
        d = t.status(); desc = d.get('description', '')
        if desc in want and desc not in seen:
            seen.add(desc); print(f"{desc:22s} {d['state']:12s} {d.get('error_message', '')}")
    for r in now:
        if os.path.exists(f'{OUT}/{r}_old.npz'): print(f'{r:12s} 캐시 이미 있음')
    print('\n둘 다 COMPLETED → ③ / 아직이면 몇 분 뒤 이 셀 다시 실행')

def process_pair(now):
    os.makedirs(f'{D}/s2', exist_ok=True)
    for r in now:
        if os.path.exists(f'{OUT}/{r}_old.npz'):
            print(f'{r:12s} 캐시 있음 — 건너뜀'); continue
        parts = sorted(glob.glob(f'{SRC_DIR}/{r}_s2old_21band*.tif')) + sorted(glob.glob(f'{D}/s2/{r}_s2old_21band*.tif'))
        if not parts:
            print(f'{r:12s} 영상이 아직 Drive에 안 보임 — 몇 분 뒤 이 셀 다시 실행'); continue
        assert len(parts) == 1, f'{r}: 파일이 여러 조각으로 나뉨 {parts} — 멈추고 보고'
        p = parts[0]
        if p.startswith(SRC_DIR):
            dst = f'{D}/s2/{os.path.basename(p)}'; shutil.move(p, dst); p = dst
        print(f'{r:12s} 영상 {os.path.getsize(p)/1e9:.2f} GB')
        build_old_cache(r, p)
        if DELETE_RAW:
            os.remove(p); print(f'{r:12s} 과거 원본 삭제 (Drive 휴지통 비우기 필요)')
    done = [r for r in now if os.path.exists(f'{OUT}/{r}_old.npz')]
    print(f'\n이 묶음 {len(done)}/{len(now)} 완료' + ('  → ④ 휴지통 비우고 다음 묶음' if len(done) == len(now) else ''))

print('함수 정의 완료')

함수 정의 완료


In [ ]:
# [3] 용량 확인
tot, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive: 전체 {tot/1e9:.1f} GB / 사용 {used/1e9:.1f} GB / 남음 {free/1e9:.1f} GB  (Colab 표시값)')
sz = {r: os.path.getsize(f'{D}/s2/{r}_s2_20band.tif') / 1e9 for r in TRAIN_REGIONS}
for r in TRAIN_REGIONS:
    print(f'  {r:12s} v1 영상 {sz[r]:.2f} GB  {"캐시 있음" if os.path.exists(f"{OUT}/{r}_old.npz") else ""}')
print(f'과거 영상은 v1과 비슷한 크기 + 1층 [추정]. 2곳 동시 최대 약 {sum(sorted(sz.values())[-2:])*1.05:.1f} GB 필요. '
      f'캐시는 지역당 약 60–90 MB 남음 [추정]')

Drive: 전체 16.1 GB / 사용 11.7 GB / 남음 4.4 GB  (Colab 표시값)
  inje         v1 영상 0.48 GB  
  hongcheon    v1 영상 1.60 GB  
  gangneung    v1 영상 0.47 GB  
  pyeongchang  v1 영상 0.49 GB  
  bonghwa      v1 영상 0.49 GB  
  geumsan      v1 영상 0.37 GB  
  geochang     v1 영상 0.50 GB  
  suncheon     v1 영상 0.58 GB  
과거 영상은 v1과 비슷한 크기 + 1층 [추정]. 2곳 동시 최대 약 2.3 GB 필요. 캐시는 지역당 약 60–90 MB 남음 [추정]


---
## 묶음 A — 인제·홍천

In [ ]:
# A-① 내보내기 제출 — 인제·홍천
NOW_A = ['inje', 'hongcheon']
submit_pair(NOW_A)

/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for UMD/hansen/global_forest_change_2024_v1_12! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by UMD/hansen/global_forest_change_2025_v1_13

Learn more: https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2024_v1_12

  warnings.warn(warning, category=DeprecationWarning)


inje         제출  장면 수 {'summer': 58, 'leafoff': 117}  Hansen global_forest_change_2024_v1_12  격자 3528x3519
hongcheon    제출  장면 수 {'summer': 88, 'leafoff': 172}  Hansen global_forest_change_2024_v1_12  격자 4420x9309

→ ② 상태 확인 (지역당 수십 분 [추정])


In [ ]:
# A-② 상태 확인 — 수시로 다시 실행. 둘 다 COMPLETED가 되면 A-③
status_pair(NOW_A)

hongcheon_s2old        COMPLETED    
inje_s2old             COMPLETED    

둘 다 COMPLETED → ③ / 아직이면 몇 분 뒤 이 셀 다시 실행


In [ ]:
# A-③ 이동 → 피처 검산 → 캐시 → 과거 원본 삭제 (지역당 몇 분)
process_pair(NOW_A)

inje         영상 0.50 GB
inje         캐시 저장 600,263픽셀, 값 있음 100.0%, 2022년 이후 산림손실 0.89%, 피처 검산 차이 0.0e+00 (65초)
inje         과거 원본 삭제 (Drive 휴지통 비우기 필요)
hongcheon    영상 1.65 GB
hongcheon    캐시 저장 599,525픽셀, 값 있음 100.0%, 2022년 이후 산림손실 1.65%, 피처 검산 차이 0.0e+00 (122초)
hongcheon    과거 원본 삭제 (Drive 휴지통 비우기 필요)

이 묶음 2/2 완료  → ④ 휴지통 비우고 다음 묶음


**A-④ Drive 휴지통 비우기** — 휴지통에 `s2old` 파일만 있는지 보고 비우기. 비운 뒤 다음 묶음으로.

---
## 묶음 B — 강릉·평창
 앞 묶음(인제·홍천)의 ③이 끝나고 휴지통을 비운 뒤 시작.

In [ ]:
# B-① 내보내기 제출 — 강릉·평창
NOW_B = ['gangneung', 'pyeongchang']
submit_pair(NOW_B)

gangneung    제출  장면 수 {'summer': 119, 'leafoff': 230}  Hansen global_forest_change_2024_v1_12  격자 3549x3557
pyeongchang  제출  장면 수 {'summer': 24, 'leafoff': 55}  Hansen global_forest_change_2024_v1_12  격자 3537x3552

→ ② 상태 확인 (지역당 수십 분 [추정])


In [ ]:
# B-② 상태 확인 — 수시로 다시 실행. 둘 다 COMPLETED가 되면 B-③
status_pair(NOW_B)

pyeongchang_s2old      COMPLETED    
gangneung_s2old        COMPLETED    

둘 다 COMPLETED → ③ / 아직이면 몇 분 뒤 이 셀 다시 실행


In [ ]:
# B-③ 이동 → 피처 검산 → 캐시 → 과거 원본 삭제 (지역당 몇 분)
process_pair(NOW_B)

gangneung    영상 0.49 GB
gangneung    캐시 저장 600,079픽셀, 값 있음 100.0%, 2022년 이후 산림손실 0.85%, 피처 검산 차이 0.0e+00 (66초)
gangneung    과거 원본 삭제 (Drive 휴지통 비우기 필요)
pyeongchang  영상 0.51 GB
pyeongchang  캐시 저장 600,530픽셀, 값 있음 100.0%, 2022년 이후 산림손실 1.94%, 피처 검산 차이 0.0e+00 (58초)
pyeongchang  과거 원본 삭제 (Drive 휴지통 비우기 필요)

이 묶음 2/2 완료  → ④ 휴지통 비우고 다음 묶음


**B-④ Drive 휴지통 비우기** — 휴지통에 `s2old` 파일만 있는지 보고 비우기. 비운 뒤 다음 묶음으로.

---
## 묶음 C — 봉화·금산
 앞 묶음(강릉·평창)의 ③이 끝나고 휴지통을 비운 뒤 시작.

In [ ]:
# C-① 내보내기 제출 — 봉화·금산
NOW_C = ['bonghwa', 'geumsan']
submit_pair(NOW_C)

bonghwa      제출  장면 수 {'summer': 86, 'leafoff': 181}  Hansen global_forest_change_2024_v1_12  격자 3551x3595
geumsan      제출  장면 수 {'summer': 35, 'leafoff': 59}  Hansen global_forest_change_2024_v1_12  격자 3262x2892

→ ② 상태 확인 (지역당 수십 분 [추정])


In [ ]:
# C-② 상태 확인 — 수시로 다시 실행. 둘 다 COMPLETED가 되면 C-③
status_pair(NOW_C)

geumsan_s2old          COMPLETED    
bonghwa_s2old          COMPLETED    

둘 다 COMPLETED → ③ / 아직이면 몇 분 뒤 이 셀 다시 실행


In [ ]:
# C-③ 이동 → 피처 검산 → 캐시 → 과거 원본 삭제 (지역당 몇 분)
process_pair(NOW_C)

bonghwa      영상 0.51 GB
bonghwa      캐시 저장 599,540픽셀, 값 있음 100.0%, 2022년 이후 산림손실 0.93%, 피처 검산 차이 0.0e+00 (63초)
bonghwa      과거 원본 삭제 (Drive 휴지통 비우기 필요)
geumsan      영상 0.38 GB
geumsan      캐시 저장 600,247픽셀, 값 있음 100.0%, 2022년 이후 산림손실 3.76%, 피처 검산 차이 0.0e+00 (46초)
geumsan      과거 원본 삭제 (Drive 휴지통 비우기 필요)

이 묶음 2/2 완료  → ④ 휴지통 비우고 다음 묶음


**C-④ Drive 휴지통 비우기** — 휴지통에 `s2old` 파일만 있는지 보고 비우기. 비운 뒤 다음 묶음으로.

---
## 묶음 D — 거창·순천
 앞 묶음(봉화·금산)의 ③이 끝나고 휴지통을 비운 뒤 시작.

In [ ]:
# D-① 내보내기 제출 — 거창·순천
NOW_D = ['geochang', 'suncheon']
submit_pair(NOW_D)

geochang     제출  장면 수 {'summer': 74, 'leafoff': 120}  Hansen global_forest_change_2024_v1_12  격자 3513x3612
suncheon     제출  장면 수 {'summer': 66, 'leafoff': 120}  Hansen global_forest_change_2024_v1_12  격자 3924x3758

→ ② 상태 확인 (지역당 수십 분 [추정])


In [ ]:
# D-② 상태 확인 — 수시로 다시 실행. 둘 다 COMPLETED가 되면 D-③
status_pair(NOW_D)

suncheon_s2old         COMPLETED    
geochang_s2old         COMPLETED    

둘 다 COMPLETED → ③ / 아직이면 몇 분 뒤 이 셀 다시 실행


In [ ]:
# D-③ 이동 → 피처 검산 → 캐시 → 과거 원본 삭제 (지역당 몇 분)
process_pair(NOW_D)

geochang     영상 0.51 GB
geochang     캐시 저장 601,114픽셀, 값 있음 100.0%, 2022년 이후 산림손실 3.24%, 피처 검산 차이 0.0e+00 (57초)
geochang     과거 원본 삭제 (Drive 휴지통 비우기 필요)
suncheon     영상 0.59 GB
suncheon     캐시 저장 600,845픽셀, 값 있음 100.0%, 2022년 이후 산림손실 1.27%, 피처 검산 차이 0.0e+00 (76초)
suncheon     과거 원본 삭제 (Drive 휴지통 비우기 필요)

이 묶음 2/2 완료  → ④ 휴지통 비우고 다음 묶음


**D-④ Drive 휴지통 비우기** — 휴지통에 `s2old` 파일만 있는지 보고 비우기.

---
## 마무리

In [ ]:
# [7] 진행표
log = json.load(open(LOG)) if os.path.exists(LOG) else {}
print(f'{"지역":12s} {"캐시":6s} {"값있음%":>7s} {"손실%":>6s}  장면 수(여름/낙엽기)')
for r in TRAIN_REGIONS:
    f = f'{OUT}/{r}_old.npz'
    if os.path.exists(f):
        z = np.load(f)
        print(f'{r:12s} {"있음":6s} {z["ok"].mean()*100:7.1f} {(z["loss"] >= 22).mean()*100:6.2f}  {log.get(r, {}).get("scenes", "")}')
    else:
        print(f'{r:12s} {"없음":6s}')
n_done = sum(os.path.exists(f'{OUT}/{r}_old.npz') for r in TRAIN_REGIONS)
print(f'\n{n_done}/8 완료' + ('  → 이 출력을 채팅에 붙여넣고 17번 실행' if n_done == 8 else ''))

지역           캐시        값있음%    손실%  장면 수(여름/낙엽기)
inje         있음       100.0   0.89  {'summer': 58, 'leafoff': 117}
hongcheon    있음       100.0   1.65  {'summer': 88, 'leafoff': 172}
gangneung    있음       100.0   0.85  {'summer': 119, 'leafoff': 230}
pyeongchang  있음       100.0   1.94  {'summer': 24, 'leafoff': 55}
bonghwa      있음       100.0   0.93  {'summer': 86, 'leafoff': 181}
geumsan      있음       100.0   3.76  {'summer': 35, 'leafoff': 59}
geochang     있음       100.0   3.24  {'summer': 74, 'leafoff': 120}
suncheon     있음       100.0   1.27  {'summer': 66, 'leafoff': 120}

8/8 완료  → 이 출력을 채팅에 붙여넣고 17번 실행
